# Simple Transfomers

In [ ]:
import pandas as pd
import glob
import os
from tqdm import tqdm
from datasets import Dataset
from simpletransformers.ner import NERModel, NERArgs
import pandas as pd
from pathlib import Path

## Preprocess Dataset

In [ ]:
def read_token_classification_file(filepath, starting_sentence_id=0):
    tokens = []
    labels = []
    sentence_ids = []

    current_sentence_id = starting_sentence_id
    temp_tokens = []
    temp_labels = []

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                if temp_tokens:
                    tokens.extend(temp_tokens)
                    labels.extend(temp_labels)
                    sentence_ids.extend([current_sentence_id] * len(temp_tokens))
                    current_sentence_id += 1
                    temp_tokens = []
                    temp_labels = []
            else:
                parts = line.split()
                if len(parts) >= 2:
                    token = parts[0]
                    label = parts[-2]  # Second-to-last column is NER tag
                    temp_tokens.append(token)
                    temp_labels.append(label)

    # Handle last sentence
    if temp_tokens:
        tokens.extend(temp_tokens)
        labels.extend(temp_labels)
        sentence_ids.extend([current_sentence_id] * len(temp_tokens))

    return tokens, labels, sentence_ids, current_sentence_id

In [ ]:
def read_all_token_classification_files(folder_path):
    all_tokens = []
    all_labels = []
    all_sentence_ids = []

    txt_files = glob.glob(os.path.join(folder_path, "*.txt"))
    sentence_id_counter = 0

    for filepath in tqdm(txt_files, colour="yellow"):
        tokens, labels, sentence_ids, sentence_id_counter = read_token_classification_file(filepath, sentence_id_counter)
        all_tokens.extend(tokens)
        all_labels.extend(labels)
        all_sentence_ids.extend(sentence_ids)

    # Create DataFrame or convert to Hugging Face Dataset
    df = pd.DataFrame({
        "sentence_id": all_sentence_ids,
        "words": all_tokens,
        "labels": all_labels
    })
    return df

In [ ]:
train_dataset = read_all_token_classification_files("./train/train")
eval_dataset = read_all_token_classification_files("./eval/eval")
train_dataset

In [ ]:
classes_dict = pd.read_csv("tag_list.csv")
classes_dict = dict(zip(classes_dict["tag"],classes_dict["class"]))

In [ ]:
def lazy_clean_labels(label):
    if label not in classes_dict.keys():
        return "O"
    return label

# If the label not appeared in tag list, then insert to "O"
train_dataset["labels"] = train_dataset["labels"].apply(lazy_clean_labels)
eval_dataset["labels"] = eval_dataset["labels"].apply(lazy_clean_labels)

## Training Model

https://simpletransformers.ai/docs/ner-specifics/

In [ ]:
check_point = f"""
/home/ai5039/.cache/huggingface/hub/models--google-bert--bert-base-multilingual-cased/snapshots/3f076fdb1ab68d5b2880cb87a0886f315b8146f8
"""

In [ ]:
model_args = {
    "num_train_epochs": 5,
    "train_batch_size": 128,
    "eval_batch_size": 128,
    "evaluate_during_training": True,
    "dataloader_num_workers": 16,
    "overwrite_output_dir": True,
    "use_early_stopping": True,
    "save_eval_checkpoints" : False,
    "save_model_every_epoch" : False,
    "save_optimizer_and_scheduler" : False,
    "manual_seed" :42,
    "max_seq_length":512
}


# Create a NERModel
model = NERModel(
    "bert",
    check_point,
    labels=list(classes_dict.keys()),
    args=model_args,
    use_cuda=True
)

In [ ]:
# Train the model
model.train_model(train_dataset, eval_data=eval_dataset)

## Inferences

https://simpletransformers.ai/docs/ner-model/

In [ ]:
# model = NERModel("bert",
#                 "outputs/best_model",
#                 labels=list(classes_dict.keys()),
#                 args=model_args,
#                 use_cuda=True)

In [ ]:
model.predict([["รัฐ" ,"ถังแตก" ,"วิก","_","7","_","สี"]],split_on_space=False)[0]

In [ ]:
def test_token_classification_file(filepath):
    tokens = []
    temp_tokens = []

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                if temp_tokens:
                    tokens.append(temp_tokens)
                    temp_tokens = []

            else:
                parts = line.split()
                if len(parts) >= 2:
                    token = parts[0]
                    temp_tokens.append(token)

    if temp_tokens:
        tokens.extend(temp_tokens)
    return tokens

In [ ]:
test_dataset = test_token_classification_files("./test/test")
test_dataset["count"] = test_dataset["words"].apply(lambda x: len(x))
test_dataset

In [ ]:
test_dataset["count"].sum()

In [ ]:
pred, logits = model.predict(test_dataset["words"],split_on_space=False)
pred[0]

## Prepare Submission

In [ ]:
pred_df = {"word" : [],"ne" : []}
count = 0
for items in tqdm(pred):
    for item in items:
        key = list(item.keys())[0]
        value = list(item.values())[0]
        if not value:
            print("empty")
        pred_df["word"].append(key)
        pred_df["ne"].append(value)
        count +=1

print(count)

In [ ]:
pred_df = pd.DataFrame(pred_df)
pred_df

In [ ]:
pred_df["ne"] = pred_df["ne"].map(classes_dict)
pred_df

In [ ]:
submission = pd.read_csv("sample_submission.csv").drop("ne", axis=1)
merged = submission.merge(pred_df, left_index=True, right_index=True, how="left")
merged.drop(["word"], axis=1, inplace=True)
merged.isnull().sum()

In [ ]:
merged.to_csv("submission.csv", index=False)